In [5]:
import psycopg2
from faker import Faker
import random
from datetime import datetime, timedelta
import pandas as pd
import os

# -----------------------------------------
# DATABASE CONNECTION
# -----------------------------------------
conn = psycopg2.connect(
    dbname="RD_Project",
    user="postgres",
    password="valeo2026",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()
fake = Faker()

# -----------------------------------------
# DATA DEFINITIONS
# -----------------------------------------

seniorities = ['Junior', 'Mid', 'Senior']

roles = [
    'Embedded Systems Engineer',
    'CO2 Reduction Engineer',
    'Software Engineer',
    'QA Engineer'
]

teams = [
    'Embedded Systems',
    'Hardware',
    'Validation',
    'Testing',
    'Software'
]

priorities = ['High','Medium','Low']

projects = [
    'Smart Grid Infrastructure',
    'Digital Transformation Phase II',
    'Renewable Energy Storage',
    'Global Supply Chain Audit',
    'Predictive Maintenance AI',
    'Autonomous Navigation Systems',
    'Cloud Migration Strategy',
    'EV Charging Network Expansion',
    'Industrial IoT Integration',
    'Cybersecurity Framework Update'
]

# -----------------------------------------
# 1. GENERATE ENGINEERS
# -----------------------------------------

print("Generating Engineers...")

for i in range(1, 51):

    cursor.execute("""
        INSERT INTO Engineers
        (engineer_id, full_name, role, seniority, team, weekly_capacity_hours)
        VALUES (%s,%s,%s,%s,%s,40)
    """,
    (
        i,
        fake.name(),
        random.choice(roles),
        random.choice(seniorities),
        random.choice(teams)
    ))

# -----------------------------------------
# 2. GENERATE PROJECTS + MILESTONES
# -----------------------------------------

print("Generating Projects & Milestones...")

project_dates = {}
milestones = []

milestone_id_counter = 1

for i, p_name in enumerate(projects,1):

    start = fake.date_between(start_date='-1y', end_date='-60d')
    deadline = start + timedelta(days=random.randint(180,365))

    project_dates[i] = start

    cursor.execute("""
        INSERT INTO Projects
        (project_id, project_name, start_date, target_deadline, priority)
        VALUES (%s,%s,%s,%s,%s)
    """,
    (
        i,
        p_name,
        start,
        deadline,
        random.choice(priorities)
    ))

    # Create 3 milestones per project
    for m in range(1,4):

        m_date = start + timedelta(days=m*60)

        cursor.execute("""
            INSERT INTO Milestones
            (milestone_id, project_id, milestone_name, target_date, is_critical_path)
            VALUES (%s,%s,%s,%s,%s)
        """,
        (
            milestone_id_counter,
            i,
            f"Phase {m} Review",
            m_date,
            True if m==3 else False
        ))

        milestones.append((milestone_id_counter,i))
        milestone_id_counter += 1


# -----------------------------------------
# 3. LOAD ENGINEERS INTO MEMORY (FASTER)
# -----------------------------------------

cursor.execute("SELECT engineer_id, seniority FROM Engineers")
engineers = dict(cursor.fetchall())

# Complexity hours reference
task_levels = {
    1:8,
    2:24,
    3:60
}

# -----------------------------------------
# 4. GENERATE TASKS
# -----------------------------------------

print("Generating Tasks...")

for t_id in range(1,1001):

    eng_id = random.randint(1,50)
    proj_id = random.randint(1,10)

    seniority = engineers[eng_id]

    # Complexity logic
    if seniority == 'Senior':
        lvl = random.choice([2,3])
    elif seniority == 'Junior':
        lvl = random.choice([1,2])
    else:
        lvl = random.choice([1,2,3])

    planned = task_levels[lvl] + random.randint(-2,5)

    # STATUS LOGIC
    prob = random.random()

    if prob < 0.55:
        status = 'Completed'
    elif prob < 0.80:
        status = 'In Progress'
    elif prob < 0.92:
        status = 'Open'
    else:
        status = 'Delayed'

    project_start = project_dates[proj_id]

    created_date = project_start + timedelta(days=random.randint(0,30))

    due_date = project_start + timedelta(days=120)

    # Assign milestone belonging to the project
    project_milestones = [m for m in milestones if m[1]==proj_id]
    milestone = random.choice(project_milestones)[0]

    if status == 'Completed':

        percent = 100
        actual_total = planned * random.uniform(0.8,1.4)
        completion_date = created_date + timedelta(days=random.randint(10,80))

    elif status == 'In Progress':

        percent = random.randint(10,90)
        actual_total = random.uniform(2,planned*1.6)
        completion_date = None

    elif status == 'Delayed':

        percent = random.randint(30,90)
        actual_total = random.uniform(planned,planned*1.8)
        completion_date = None

    else:

        percent = 0
        actual_total = 0
        completion_date = None

    cursor.execute("""
        INSERT INTO Tasks
        (task_id, project_id, milestone_id, assigned_to,
        task_level_id, task_status, planned_hours,
        percent_complete, created_date, due_date, completion_date)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """,
    (
        t_id,
        proj_id,
        milestone,
        eng_id,
        lvl,
        status,
        planned,
        percent,
        created_date,
        due_date,
        completion_date
    ))

    # -----------------------------------------
    # 5. GENERATE TIME LOGS
    # -----------------------------------------

    remaining_hours = actual_total
    log_date = created_date + timedelta(days=2)

    while remaining_hours > 0:

        daily = min(remaining_hours, random.uniform(1,7))

        cursor.execute("""
            INSERT INTO Time_Logs
            (task_id, engineer_id, log_date, hours_spent)
            VALUES (%s,%s,%s,%s)
        """,
        (
            t_id,
            eng_id,
            log_date,
            round(daily,2)
        ))

        remaining_hours -= daily

        gap = random.randint(1,3)
        log_date += timedelta(days=gap)

        if log_date > datetime.now().date():
            break

# -----------------------------------------
# COMMIT DATABASE
# -----------------------------------------

conn.commit()

print("Database populated successfully!")

# -----------------------------------------
# 6. EXPORT CSV FOR POWER BI
# -----------------------------------------

print("Exporting CSV files...")

target_folder = r"C:\Users\Lenovo\Documents\Data_Project"

if not os.path.exists(target_folder):
    os.makedirs(target_folder)

tables = [
    'Engineers',
    'Projects',
    'Milestones',
    'Tasks',
    'Time_Logs'
]

for table in tables:

    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)

    file_path = os.path.join(target_folder, f"{table}.csv")

    df.to_csv(file_path,index=False)

print("CSV files exported!")

# -----------------------------------------
# CLOSE CONNECTION
# -----------------------------------------

cursor.close()
conn.close()

print("Process completed successfully.")

Generating Engineers...
Generating Projects & Milestones...
Generating Tasks...
Database populated successfully!
Exporting CSV files...
CSV files exported!
Process completed successfully.


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17032\3521622269.py:296: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {table}", conn)


In [7]:
import psycopg2
import pandas as pd
import os

conn = psycopg2.connect(
    dbname="RD_Project",
    user="postgres",
    password="valeo2026",
    host="localhost",
    port="5432"
)

target_folder = r"C:\Users\Lenovo\Documents\Data_Project"

tables = [
    'Engineers',
    'Projects',
    'Task_Levels',
    'Milestones',
    'Tasks',
    'Time_Logs'
]

for table in tables:

    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)

    file_path = os.path.join(target_folder, f"{table}.csv")

    df.to_csv(file_path,index=False)

print("CSV files exported!")

conn.close()

CSV files exported!


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17032\612402661.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
